In [13]:
import os
#print(pwd)
os.chdir('../')
%pwd


'e:\\Text_Summarizer'

In [14]:
#entity
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [15]:
#configuration
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [16]:

class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH):
        
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)-> DataIngestionConfig:

        config=self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [17]:
#component
import os
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [18]:
class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config= config

    def download_file(self):

        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} download! with following inf: \n{headers}")

        else:
            logger.info(f"file already exists of size : {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        Extracting the zipfile into data directiory
        """
        unzip_path=self.config.unzip_dir

        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)

            

In [19]:
#pipeline

try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-04-19 16:09:46,033: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-04-19 16:09:46,036: INFO: common: yaml file: params.yaml loaded successfully]
[2025-04-19 16:09:46,038: INFO: common: created directory at: artifacts]
[2025-04-19 16:09:46,039: INFO: common: created directory at: artifacts/data_ingestion]


[2025-04-19 16:09:47,443: INFO: 3323624285: artifacts/data_ingestion/data.zip download! with following inf: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: CD9C:94603:36B7D6:83460C:68037D72
Accept-Ranges: bytes
Date: Sat, 19 Apr 2025 10:39:47 GMT
Via: 1.1 varnish
X-Served-By: cache-maa10241-MAA
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1745059187.909614,VS0,VE717
Vary: Authorization,Accept-Encoding,Origin
Access-Control-Allow-Origin: *
Cross-Origin-Resource-Policy: cross-origin
X-Fastly-Request-ID: 0828c35688e45c37647b55339f3d82e4ba4694e6
Expires: Sat, 19 Apr 2025 10:44:47 GMT
Source-Age: 0

]
